# Mathematical Engineering — Financial Engineering, FY 2025-2026

## Buy Side, Lab 4b — Risk-Based Allocation Strategies

This notebook implements and analyses risk-based allocation strategies on the Euro Stoxx 50 universe with monthly rebalancing and a 2-year rolling estimation window.

**Outline** 

0. Setup & rolling estimators
1. Part I — Equal Risk Contribution (ERC)
2. Part II — Hierarchical Risk Parity (HRP):
   1. Detoning intuition
   2. Cluster stability across rebalances (Rand index)
   3. Sensitivity to clustering choice and to the volatility allocation scheme
3. Part III — Connecting the dots: ERC vs HRP head-to-head and the role of shrinkage


## 0. Setup


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as sch
from sklearn.metrics import rand_score

from utilities.backtest import backtest, portfolio_returns
from utilities.covariance_utilities import (
    prepare_rolling_estimation_window,
    covariance_to_correlation,
    risk_contribution,
)
from utilities.hierarchical_clustering import hierarchical_clustering
from utilities.hierarchical_risk_parity import (
    correlation_to_hrp_distance,
    dendrogram_iteration,
    hierarchical_risk_parity,
    recursive_bisection,
)
from utilities.portfolio_optimization import (
    equal_risk_contribution_portfolio,
    inverse_volatility_portfolio,
)
from utilities.principal_component_analysis import (
    detone,
    principal_component_analysis,
)
from utilities.shrinkage import constant_corr_shrinkage, market_factor_shrinkage


In [ ]:
# Read prices, compute daily simple returns
last_prices = pd.read_csv("data/sx5e_underlyings.csv", index_col="Date", parse_dates=True
)
performance = last_prices.pct_change().iloc[1:]


In [3]:
# Estimation parameters
estimation_window = 252 * 2  # mistake : 2 years 
min_coverage = 0.95  # asset must have >=95% non-NaN in the window
trading_days = 252 

# Rebalance on the last trading day of each month
rebalance_dates = pd.DatetimeIndex(
    performance.groupby(pd.Grouper(freq="ME"))
    .apply(lambda x: x.index[-1] if len(x) > 0 else None)
    .dropna()
    .values
)


## 1. Pre-computation: rolling covariance estimators

For each rebalance date we estimate three covariance matrices on the trailing 2-year window:

- **sample** — plain `cov()`,
- **constant_corr** — Ledoit–Wolf (2003) constant-correlation shrinkage,
- **mkt_factor** — Ledoit–Wolf (2002) single-factor shrinkage with the
  equally-weighted return of the surviving universe used as the market
  proxy.

The asset universe at every rebalance is restricted to names with
≥95% non-missing observations in the window.


In [4]:
covariances = {}  # date -> {estimator_name: cov_df}
universes = {}  # date -> [tickers]

for rebalance_date in rebalance_dates:
    cur_returns, diag = prepare_rolling_estimation_window(
        returns=performance,
        rebalance_date=rebalance_date,
        lookback=estimation_window,
        min_coverage=min_coverage,
        return_diagnostics=True,
    )
    if diag["row_count"] < estimation_window or cur_returns.shape[1] == 0:
        continue

    reb = rebalance_date.to_pydatetime().date()

    universes[reb] = cur_returns.columns.tolist()  
    covariances[reb] = {
        "sample": cur_returns.cov().to_numpy(),
        "constant_corr": constant_corr_shrinkage(cur_returns)["target"].to_numpy(),
        "mkt_factor": market_factor_shrinkage(cur_returns, cur_returns.iloc[:, -1])["target"].to_numpy(),
    }

all_dates = sorted(covariances.keys())

## Part I — Equal Risk Contribution (ERC)


### 1.b Sanity check: Risk contribution dispersion & IV ≡ ERC under a diagonal Σ


In [24]:
iv_erc_delta = {}
erc_normalized_range = {}

for date in all_dates:
    iv_w  = inverse_volatility_portfolio(covariances[date]["sample"])
    erc_w = equal_risk_contribution_portfolio(covariances[date]["sample"], pcr_tolerance=0.05, options={"maxiter": 5000, "ftol": 1e-15, "disp": False})

    iv_rc  = risk_contribution(iv_w,  covariances[date]["sample"])
    erc_rc = risk_contribution(erc_w, covariances[date]["sample"])
        
    iv_erc_delta[date]        = iv_rc - erc_rc
    erc_normalized_range[date] = (erc_rc.max() - erc_rc.min()) / erc_rc.mean()

# ERC dispersion
erc_ranges = pd.Series(erc_normalized_range)
print("=== ERC Normalized Range (max-min)/mean ===")
print(erc_ranges.describe())

# delta IV - ERC 
delta_df = pd.DataFrame(
    {date: pd.Series(delta, index=universes[date]) 
     for date, delta in iv_erc_delta.items()}
).T  # dates x assets
print("\n=== IV - ERC Risk Contributions (time-averaged) ===")
print(delta_df.mean().describe())

=== ERC Normalized Range (max-min)/mean ===
count    99.000000
mean      0.607105
std       0.223829
min       0.224877
25%       0.422942
50%       0.593805
75%       0.851548
max       0.994385
dtype: float64

=== IV - ERC Risk Contributions (time-averaged) ===
count    50.000000
mean     -0.000239
std       0.002136
min      -0.008759
25%      -0.000779
50%       0.000220
75%       0.000903
max       0.001958
dtype: float64


In [25]:
# Build a diagonal covariance matrix from the sample variances
date = all_dates[0]

S = covariances[date]["sample"]
diag_cov = np.diag(np.diag(S))  # zero out all off-diagonal elements

iv_w_diag  = inverse_volatility_portfolio(diag_cov)
erc_w_diag = equal_risk_contribution_portfolio(diag_cov, pcr_tolerance=0.01)

print(np.allclose(iv_w_diag, erc_w_diag, atol=1e-4))

True


### 1.c ERC backtest across the three covariance estimators


In [26]:
# 1. Compute ERC portfolios for each estimator
erc_portfolios = {"sample": {}, "constant_corr": {}, "mkt_factor": {}}

for reb in all_dates:
    universe = universes[reb]
    for est_name, cov in covariances[reb].items():
        try:
            w = equal_risk_contribution_portfolio(
                cov,
                pcr_tolerance=0.05,
                options={"maxiter": 5000, "ftol": 1e-15, "disp": False},
            )
        except ValueError:
            w = equal_risk_contribution_portfolio(cov, ignore_objective=True)
        erc_portfolios[est_name][reb] = pd.Series(w, index=universe)

# 2. Convert to DataFrames (dates x assets)
erc_weights = {
    est: pd.DataFrame(weights_dict).T  # dates x assets
    for est, weights_dict in erc_portfolios.items()
}

# 3. Backtest using your existing functions
erc_cumrets = {
    est: backtest(erc_weights[est], performance)
    for est in erc_weights
}
erc_returns = {
    est: portfolio_returns(erc_weights[est], performance)
    for est in erc_weights
}

# 4. Metrics
def turnover(weights_df: pd.DataFrame) -> pd.Series:
    """One-way turnover at each rebalance date."""
    return weights_df.diff().abs().sum(axis=1).dropna()

summary = {}
for est in erc_weights:
    ret  = erc_returns[est]
    cum  = erc_cumrets[est]
    dd   = (cum - cum.cummax()) / cum.cummax()
    vol  = ret.ewm(span=63).std() * np.sqrt(252)
    to   = turnover(erc_weights[est])

    summary[est] = {
        "Total Return (%)":  (cum.iloc[-1] - 1) * 100,
        "Avg EWM Vol (%)":   vol.mean() * 100,
        "Max Drawdown (%)":  dd.min() * 100,
        "Avg Turnover (%)":  to.mean() * 100,
    }

print(pd.DataFrame(summary).round(2))

                  sample  constant_corr  mkt_factor
Total Return (%)  163.83         163.02      160.55
Avg EWM Vol (%)    16.82          16.95       16.78
Max Drawdown (%)  -39.14         -39.30      -38.95
Avg Turnover (%)    1.89           1.36        2.26


## Part II — Hierarchical Risk Parity (HRP)


### 2.a Detoning — intuition


In [ ]:
# !!! COMPLETE AS APPROPRIATE !!!

### 2.c Cluster stability across rebalances (Rand index)

We measure how stable the clustering is between consecutive rebalances via the Rand index. A value close to 1 means the partition into $k$ clusters changes little month-to-month.


In [ ]:
LINKAGE_METHOD = "ward"
DIST_METRIC = "euclidean"
N_CLUSTERS_GRID = [5, 8, 10, 15]
N_CLUSTERS_FOCUS = 10
COV_KEYS = ["sample", "constant_corr", "mkt_factor"]
DETONE_KEYS = [False, True]


linkages = {(c, d): {} for c in COV_KEYS for d in DETONE_KEYS}
clusters = {
    (c, d, k): {} for c in COV_KEYS for d in DETONE_KEYS for k in N_CLUSTERS_GRID
}

for reb in all_dates:
    for cov_key in COV_KEYS:
        cov = covariances[reb][cov_key]
        for detone_flag in DETONE_KEYS:
            link = None  # !!! COMPLETE AS APPROPRIATE !!!
            linkages[(cov_key, detone_flag)][reb] = link
            for n_clusters in N_CLUSTERS_GRID:
                clusters[(cov_key, detone_flag, n_clusters)][reb] = pd.Series(
                    sch.cut_tree(link.values, n_clusters=n_clusters).flatten(),
                    index=cov.index,
                )

# !!! COMPLETE AS APPROPRIATE !!!

### 2.d Sensitivity of HRP weights — clustering vs. allocation traversal


In [ ]:
LINK_GRID = ["single", "ward", "complete", "average"]
ALLOC_GRID = {
    "recursive_bisection": recursive_bisection,
    "dendrogram_iteration": dendrogram_iteration,
}

hrp_grid = {}
for cov_key in COV_KEYS:
    for link in LINK_GRID:
        for alloc_name, alloc in ALLOC_GRID.items():
            for d in DETONE_KEYS:
                hrp_grid[(cov_key, link, alloc_name, d)] = (
                    None  # !!! COMPLETE AS APPROPRIATE !!!
                )

# !!! COMPLETE AS APPROPRIATE !!!

## Part III — Connecting the Dots


### 3.a ERC vs HRP — Robustness Against Estimator Choice

In [ ]:
# !!! COMPLETE AS APPROPRIATE !!!

### 3.b ERC vs HRP — performance, vol, drawdown, turnover


In [ ]:
# !!! COMPLETE AS APPROPRIATE !!!